# METRIC trained-model deconvolution workflow

This notebook applies one or more trained METRIC LDA models to an independent methylation test set.

It is a cleaned, general-purpose version of the original experimental notebook. It removes project-specific absolute paths and uses relative paths so that the notebook can be placed in the same directory as `train_model.py` and `gensim_Utils.py`.

## Expected repository layout

This notebook is assumed to be saved in the same directory as `train_model.py`, for example:

```text
METRIC-RRBS/
├── Code_batch/
│   ├── train_model.py
│   ├── gensim_Utils.py
│   └── trained_model_deconv.ipynb
├── input/
│   ├── groupN2.csv
│   ├── Merged_Markers_top250_U_deldup.bed
│   ├── external_test_countU.bed
│   └── external_test_sample_group.csv
└── results/
    └── groupN2_top250_U_TrainSet_countU_top50/
        └── topic_37/
            ├── ModelTrained/
            ├── topic37_topword50_marker_topic_mat.txt
            └── topic37_topword50_topic_celltype_mat.txt
```

The notebook uses the parent directory of its current working directory as the project root, so `../results` points to the trained-model results generated by `train_model.py`.

In [1]:
from pathlib import Path
import fnmatch
import os
import sys

import gensim
import numpy as np
import pandas as pd
import scipy
import scipy.sparse
from gensim import models

# The notebook should be placed in the same folder as train_model.py and gensim_Utils.py.
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent.resolve()

# Make local utility imports work when the notebook is launched from this directory.
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

try:
    from gensim_Utils import generate_test_corpus
except ImportError:
    # Backward compatibility for older code releases.
    from gensim_Utils import generate_test_corpus_v2 as generate_test_corpus

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Project root:       {PROJECT_ROOT}")

Notebook directory: /data_center_02/project/SR/sunxiaojun/Deconv/METRIC/METRIC_release/METRIC-RRBS/Code
Project root:       /data_center_02/project/SR/sunxiaojun/Deconv/METRIC/METRIC_release/METRIC-RRBS


## 1. Configure paths and model parameters

Edit only this cell for most applications.

Required input files:

- `TEST_SET_FILE`: a count-U matrix in BED-like format, using the same marker coordinate convention as the training atlas.
- `TEST_SAMPLE_GROUP_FILE`: a CSV file with at least one column named `name`; these names must match the sample columns in the test matrix.
- `TRAIN_MARKER_INFO_FILE`: the marker information file used during model training.
- `MODEL_RESULTS_DIR`: the `results/` directory produced by `train_model.py`.

In [2]:
# Dataset/model identifiers used during training.
SELECT_ID = "groupN2"
TOPN = "250_U"

# Use a single optimized model by default.
# To deconvolve with all models from a parameter grid, replace these with the full lists used for training.
N_TOP_WORDS_LIST = [30]
N_TOPICS_LIST = [27]

# Input and model directories.
INPUT_DIR = PROJECT_ROOT / "input"
MODEL_RESULTS_DIR = PROJECT_ROOT / "results"

# Independent test-set files.
# Replace these placeholder names with your own independent test set files.
TEST_SET_FILE = INPUT_DIR / "external_test_countU.bed"
TEST_SAMPLE_GROUP_FILE = INPUT_DIR / "external_test_sample_group.csv"

# Output directory for trained-model deconvolution.
OUTPUT_DIR = PROJECT_ROOT / "results_trainedModel_deconv" / "external_testset"

# Optional: set to True to skip model combinations with incomplete files.
SKIP_MISSING_MODELS = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

Output directory: /data_center_02/project/SR/sunxiaojun/Deconv/METRIC/METRIC_release/METRIC-RRBS/results_trainedModel_deconv/external_testset


## 2. Helper functions

These functions locate the trained model, load marker information, generate the test corpus and compute sample-level cell-type fractions.

In [3]:
def resolve_first_existing(paths):
    """Return the first existing path from a list of candidate paths."""
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    candidates = "\n".join(str(Path(p)) for p in paths)
    raise FileNotFoundError(f"None of the candidate files exist:\n{candidates}")


def get_train_prefix(select_id, topn):
    """Return the training prefix used by train_model.py."""
    return f"{select_id}_top{topn}_TrainSet_countU"


def find_marker_info_file(input_dir, topn):
    """Find the marker information file, allowing for Top/top case differences."""
    return resolve_first_existing([
        input_dir / f"Merged_Markers_top{topn}_deldup.bed",
        input_dir / f"Merged_Markers_Top{topn}_deldup.bed",
    ])


def find_model_dir(model_results_dir, train_prefix, n_top_words, n_topics):
    """Return the model directory created by train_model.py."""
    return model_results_dir / f"{train_prefix}_top{n_top_words}" / f"topic_{n_topics}"


def find_single_model_file(model_dir):
    """Find the saved gensim LDA model file in ModelTrained/."""
    model_path = model_dir / "ModelTrained"
    model_files = sorted(model_path.glob("*.model"))
    if not model_files:
        raise FileNotFoundError(f"No *.model file found in {model_path}")
    return model_files[0]


def find_marker_topic_file(model_dir, n_topics, n_top_words):
    """Find the marker-topic matrix file for a trained model."""
    pattern = f"topic{n_topics}_topword{n_top_words}_marker_topic_mat*.txt"
    files = sorted(model_dir.glob(pattern))
    if not files:
        raise FileNotFoundError(
            f"No marker-topic matrix matching {pattern} was found in {model_dir}"
        )
    return files[0]


def find_topic_celltype_file(model_dir, n_topics, n_top_words):
    """Find the topic-cell-type matrix file for a trained model."""
    path = model_dir / f"topic{n_topics}_topword{n_top_words}_topic_celltype_mat.txt"
    if not path.exists():
        raise FileNotFoundError(f"Topic-cell-type matrix not found: {path}")
    return path

def read_sample_group(sample_group_file):
    """
    Read sample metadata for external deconvolution.

    Required column:
        name: sample names matching the columns in the countU matrix.

    Optional columns:
        group: known sample group or cell type label.
        actual: ground-truth proportion, if available for simulated mixtures.
    """
    sample_group = pd.read_csv(sample_group_file)

    if "name" not in sample_group.columns:
        raise ValueError("Sample group file must contain a 'name' column.")

    if "group" not in sample_group.columns:
        sample_group["group"] = "Unknown"

    if "actual" not in sample_group.columns:
        sample_group["actual"] = np.nan

    sample_group["name"] = sample_group["name"].astype(str)
    sample_group["group"] = sample_group["group"].astype(str)

    return sample_group

def read_feature_markers(marker_topic_file):
    """
    Read the marker-topic matrix and return the training marker order.

    The marker-topic file generated by train_model.py stores markers as the row index.
    Keeping this order is important because the trained model and test corpus must use
    the same marker order.
    """
    marker_topic_df = pd.read_csv(marker_topic_file, sep="\t", index_col=0)
    feature_markers = marker_topic_df.index.astype(str).tolist()
    if len(feature_markers) == 0:
        raise ValueError(f"No markers were found in {marker_topic_file}")
    return feature_markers


def load_sample_names(sample_group_file):
    """Read the test sample metadata file and return sample names."""
    sample_group = pd.read_csv(sample_group_file)
    if "name" not in sample_group.columns:
        raise ValueError(f"{sample_group_file} must contain a column named 'name'.")
    return sample_group["name"].astype(str).tolist(), sample_group

In [4]:
def deconvolve_with_trained_model(
    test_set_file,
    test_sample_group_file,
    train_marker_info_file,
    model_dir,
    n_topics,
    n_top_words,
    output_dir,
):
    """
    Apply one trained METRIC model to an independent test set.

    Parameters
    ----------
    test_set_file : str or Path
        BED-like count-U matrix for the independent test set.
    test_sample_group_file : str or Path
        CSV file containing at least a 'name' column matching test-set sample columns.
    train_marker_info_file : str or Path
        Marker information file used during training.
    model_dir : str or Path
        Directory containing ModelTrained/, marker-topic matrix and topic-cell-type matrix.
    n_topics : int
        Number of topics used by the trained model.
    n_top_words : int
        Number of top markers used by the trained model.
    output_dir : str or Path
        Directory where output files will be written.

    Returns
    -------
    pd.DataFrame
        Normalized sample-by-cell-type deconvolution result.
    """
    test_set_file = Path(test_set_file)
    test_sample_group_file = Path(test_sample_group_file)
    train_marker_info_file = Path(train_marker_info_file)
    model_dir = Path(model_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    marker_topic_file = find_marker_topic_file(model_dir, n_topics, n_top_words)
    topic_celltype_file = find_topic_celltype_file(model_dir, n_topics, n_top_words)
    model_file = find_single_model_file(model_dir)

    feature_markers = read_feature_markers(marker_topic_file)
    sample_names, sample_group_df = load_sample_names(test_sample_group_file)

    print(f"[INFO] Loading model: {model_file}")
    lda_trained = models.LdaModel.load(str(model_file))

    print("[INFO] Generating test corpus...")
    test_sets = generate_test_corpus(
        file_path=str(test_set_file),
        direction="U",
        marker_info_file=str(train_marker_info_file),
        sample_group_file=str(test_sample_group_file),
        feature_markers=feature_markers,
    )

    # Use minimum_probability=0.0 so that all topics are retained in the output matrix.
    topic_test_sample = lda_trained.get_document_topics(
        test_sets["corpus"],
        minimum_probability=0.0,
    )
    topic_test_sample_mat = gensim.matutils.corpus2csc(
        topic_test_sample,
        num_terms=n_topics,
    )

    # Save topic-by-sample matrix.
    topic_test_sample_npz = output_dir / f"topic{n_topics}_topword{n_top_words}_topic_testSample_mat.npz"
    topic_test_sample_txt = output_dir / f"topic{n_topics}_topword{n_top_words}_topic_testSample_mat.txt"
    scipy.sparse.save_npz(topic_test_sample_npz, topic_test_sample_mat)

    topic_test_sample_df = pd.DataFrame(
        topic_test_sample_mat.todense(),
        index=[f"Topic {i}" for i in range(1, topic_test_sample_mat.shape[0] + 1)],
        columns=sample_names,
    )
    topic_test_sample_df.to_csv(topic_test_sample_txt, sep="\t")

    # Read the learned topic-to-cell-type mapping from the training output.
    topic_celltype_df = pd.read_csv(topic_celltype_file, sep="\t", index_col=0)
    celltype_topic_df = topic_celltype_df.transpose()

    # Convert sample-topic distributions into sample-cell-type estimates.
    topic_matrix = topic_test_sample_mat.toarray().transpose()
    celltype_matrix = celltype_topic_df.iloc[:, :topic_matrix.shape[1]].transpose().to_numpy()
    sample_celltype = np.dot(topic_matrix, celltype_matrix)

    celltype_names = celltype_topic_df.index.astype(str).tolist()
    sample_celltype_df = pd.DataFrame(
        sample_celltype,
        index=sample_names,
        columns=celltype_names,
    )

    unnormalized_file = output_dir / f"topic{n_topics}_topword{n_top_words}_testSample_celltype_unnormalized.tsv"
    sample_celltype_df.to_csv(unnormalized_file, sep="\t")

    # Row-normalize to obtain relative cell-type proportions per sample.
    row_sums = sample_celltype.sum(axis=1, keepdims=True)
    sample_celltype_norm = np.divide(
        sample_celltype,
        row_sums,
        out=np.zeros_like(sample_celltype, dtype=float),
        where=row_sums != 0,
    )

    sample_celltype_norm_df = pd.DataFrame(
        sample_celltype_norm,
        index=sample_names,
        columns=celltype_names,
    )
    sample_celltype_norm_df["topic"] = int(n_topics)
    sample_celltype_norm_df["topword"] = int(n_top_words)

    normalized_file = output_dir / f"topic{n_topics}_topword{n_top_words}_testSample_celltype_fraction.tsv"
    sample_celltype_norm_df.to_csv(normalized_file, sep="\t")

    print(f"[OK] Saved normalized deconvolution result: {normalized_file}")
    return sample_celltype_norm_df

## 3. Validate required input files

Run this cell before deconvolution. If any file is missing, update the paths in the configuration cell.

In [7]:
TRAIN_PREFIX = get_train_prefix(SELECT_ID, TOPN)
TRAIN_MARKER_INFO_FILE = find_marker_info_file(INPUT_DIR, TOPN)

required_files = {
    "TRAIN_MARKER_INFO_FILE": TRAIN_MARKER_INFO_FILE,
    "TEST_SET_FILE": TEST_SET_FILE,
    "TEST_SAMPLE_GROUP_FILE": TEST_SAMPLE_GROUP_FILE,
}

for label, path in required_files.items():
    path = Path(path)
    print(f"{label}: {path}")
    if not path.exists():
        raise FileNotFoundError(f"{label} does not exist: {path}")

print("[OK] Required input files were found.")

TRAIN_MARKER_INFO_FILE: /data_center_02/project/SR/sunxiaojun/Deconv/METRIC/METRIC_release/METRIC-RRBS/input/Merged_Markers_top250_U_deldup.bed
TEST_SET_FILE: /data_center_02/project/SR/sunxiaojun/Deconv/METRIC/METRIC_release/METRIC-RRBS/input/external_test_countU.bed
TEST_SAMPLE_GROUP_FILE: /data_center_02/project/SR/sunxiaojun/Deconv/METRIC/METRIC_release/METRIC-RRBS/input/external_test_sample_group.csv
[OK] Required input files were found.


## 4. Run deconvolution with one or multiple trained models

By default, this runs the optimized model defined by `N_TOP_WORDS_LIST = [50]` and `N_TOPICS_LIST = [37]`.

To run a full parameter grid, replace these lists with the parameter values used during training.

In [8]:
all_results = []

for n_top_words in N_TOP_WORDS_LIST:
    for n_topics in N_TOPICS_LIST:
        print(f"\n=== Deconvolving with n_top_words={n_top_words}, n_topics={n_topics} ===")

        model_dir = find_model_dir(
            MODEL_RESULTS_DIR,
            TRAIN_PREFIX,
            n_top_words=n_top_words,
            n_topics=n_topics,
        )
        per_model_output_dir = OUTPUT_DIR / f"{TRAIN_PREFIX}_top{n_top_words}" / f"topic_{n_topics}"

        try:
            result_df = deconvolve_with_trained_model(
                test_set_file=TEST_SET_FILE,
                test_sample_group_file=TEST_SAMPLE_GROUP_FILE,
                train_marker_info_file=TRAIN_MARKER_INFO_FILE,
                model_dir=model_dir,
                n_topics=int(n_topics),
                n_top_words=int(n_top_words),
                output_dir=per_model_output_dir,
            )
            all_results.append(result_df)

        except Exception as exc:
            message = f"[ERROR] Failed for n_top_words={n_top_words}, n_topics={n_topics}: {exc}"
            if SKIP_MISSING_MODELS:
                print(message)
                continue
            raise

if not all_results:
    raise RuntimeError("No deconvolution results were generated. Please check the model and input paths.")

final_results = pd.concat(all_results, axis=0)
final_results_file = OUTPUT_DIR / "final_trained_model_deconvolution_results.tsv"
final_results.to_csv(final_results_file, sep="\t")

print(f"\n[OK] Combined result saved to: {final_results_file}")
final_results.head()


=== Deconvolving with n_top_words=30, n_topics=27 ===
[INFO] Loading model: /data_center_02/project/SR/sunxiaojun/Deconv/METRIC/METRIC_release/METRIC-RRBS/results/groupN2_top250_U_TrainSet_countU_top30/topic_27/ModelTrained/groupN2_top250_U_TrainSet_countU_lda.model
[INFO] Generating test corpus...
[ERROR] Failed for n_top_words=30, n_topics=27: Sample group file is missing required columns: ['group']


RuntimeError: No deconvolution results were generated. Please check the model and input paths.

## 5. Optional: select one model result for downstream analysis

If multiple models were evaluated, this cell extracts one model combination for downstream visualization or classification.

In [ ]:
SELECT_TOPWORD = 50
SELECT_TOPIC = 37

selected_result = final_results[
    (final_results["topword"] == SELECT_TOPWORD) &
    (final_results["topic"] == SELECT_TOPIC)
].copy()

selected_result_file = OUTPUT_DIR / f"selected_topword{SELECT_TOPWORD}_topic{SELECT_TOPIC}_deconvolution.tsv"
selected_result.to_csv(selected_result_file, sep="\t")

print(f"[OK] Selected model result saved to: {selected_result_file}")
selected_result.head()

## Notes

1. This notebook assumes that the independent test matrix has already been converted into the same count-U feature format used by METRIC.
2. The test sample metadata file must contain a `name` column.
3. The marker order is taken from the trained model's marker-topic matrix, ensuring that the test corpus uses the same marker order as the training corpus.
4. Outputs are saved under `../results_trainedModel_deconv/`, relative to the notebook directory.